# Translate 2-bit Quark models with OGA (onnxruntime-genai)

Walkthrough of translating a **standard-Quark uint2** checkpoint into an ONNX Runtime
`MatMulNBits` model (bits=2, float zero-point) using the onnxruntime-genai model builder.

**Prerequisites**
- onnxruntime-genai with 2-bit support (`--precision int2`): branch `2bit_upstream_qad`
  for plain QAD uint2 models, or `2bit_upstream_lora_kd` for LoRA+KD models.
- onnxruntime >= 1.27 (float zero-point 2-bit `MatMulNBits`).
- A standard-Quark uint2 model (see [../README_QAD_2BIT.md](../README_QAD_2BIT.md)).

## 1. QAD (dense) uint2 -> MatMulNBits

Set the paths for your Quark uint2 model and the desired ONNX output directory.


In [ ]:
QUARK_UINT2_MODEL = "/root/models/phi4_qad_v6_quark_uint2_native"  # standard-Quark uint2 export
OGA_OUT = "/root/oga_out/phi4_qad_uint2"
CACHE = "/root/oga_cache"

In [ ]:
# Translate with the OGA model builder: 2-bit weights become MatMulNBits (bits=2, float ZP).
!python -m onnxruntime_genai.models.builder \
    -i $QUARK_UINT2_MODEL -o $OGA_OUT -p int2 -e cpu -c $CACHE

Expected: `model.onnx` (+ `model.onnx.data`), `genai_config.json`, tokenizer files in `OGA_OUT`.
Weight matmuls are emitted as `MatMulNBits` with `bits=2` and a float zero-point.


## 2. Sanity-check generation


In [ ]:
import onnxruntime_genai as og

model = og.Model(OGA_OUT)
tok = og.Tokenizer(model)
ids = tok.encode("The capital of France is")
gen = og.Generator(model, og.GeneratorParams(model))
gen.append_tokens(ids)
out = []
for _ in range(20):
    gen.generate_next_token()
    if gen.is_done():
        break
    out.append(gen.get_next_tokens()[0])
print(tok.decode(out))

Expected (phi-4 QAD v6): ` Paris. The city is located in the northwestern region ...`


## 3. LoRA+KD models

LoRA+KD checkpoints translate with the same builder command. The shipped export is a
standard `Phi3ForCausalLM` (or `Qwen2ForCausalLM`) whose `model.safetensors` additionally
carries, per quantized projection, an `input_prescale` vector and one shared
`shared_input_rotation_<in>` matrix per input size, plus a separate PEFT adapter under
`lora_adapters/`. The builder detects these **structurally** (no special architecture name is
required) and bakes them into the graph:

- **Factored online rotation** `(x * input_prescale) @ shared_input_rotation` feeds each
  `MatMulNBits`. The rotation matrix is emitted once per input size and shared across
  projections of that size.
- **Additive LoRA** `(lora_B @ lora_A @ x) * scaling` is added to the projection output;
  `lora_A` is shared across split projections (q/k/v, gate/up) and `lora_B` is split along
  the output dim.

```bash
python -m onnxruntime_genai.models.builder -i <lora_kd_uint2_model> -o <out> -p int2 -e cpu
```

> Ships on branch `2bit_upstream_lora_kd`. Validated on a phi-4 LoRA+KD uint2 export
> (WikiText-2 test PPL 8.23; 40 x 2048-token chunks, ONNX Runtime CPU).

## Next: add INT16 activation Q-DQ

Feed `OGA_OUT` into [add_int16_qdq.ipynb](add_int16_qdq.ipynb) (or `add_int16_qdq.py`) to produce a W2A16 model.
